In [ ]:
import sys
import os

# 设置你的 main.py 所在目录路径，例如：
project_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"

# 加入到 sys.path（如果尚未添加）
if project_dir not in sys.path:
    sys.path.append(project_dir)

# 检查是否添加成功
print("Updated sys.path:", sys.path)

Updated sys.path: ['/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/scripts', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python39.zip', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/lib-dynload', '', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/site-packages', '/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/']


In [ ]:
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import (
    prune_wanda,
    check_sparsity,
    get_mask,
    prune_wandg_set_difference,
)
from lib.model_wrapper import prune_wanda_v2, prune_wandg
from lib.eval import eval_ppl, eval_zero_shot, eval_attack
from vllm import LLM
import argparse

In [ ]:
# 📌 设置参数（你原本命令行中给出的内容）
# 构造参数

model="llama2-7b-chat-hf"
method="wanda"
sparsity_type="unstructured"
suffix="weightonly"
save_dir= f"out/{model}/{sparsity_type}/{method}_{suffix}/GSM8K_direct"
os.makedirs(save_dir, exist_ok=True)

args = argparse.Namespace(
    model=model,
    model_base="llama2-7b-hf",
    seed=0,
    nsamples=128,
    sparsity_ratio=0.1,
    sparsity_type=sparsity_type,
    prune_method=method,
    prune_data="GSM8K_direct",
    use_diff=False,
    neg_prune=False,
    recover_from_base=False,
    p=0.5,
    q=0.5,
    top_k_heads=10,
    cache_dir="llm_weights",
    use_variant=False,
    save=save_dir,
    save_model=None,
    save_mask=None,
    dump_wanda_score=False,
    eval_zero_shot=True,
    eval_attack=True,
    save_attack_res=True,
    prune_part=False,
    disentangle=True,  # 注意：原 argparse 中是 --entangle_prompt_feat -> dest="disentangle", action="store_false"
    decouple_align_utility=False,
    decouple_align_misalign=False,
    rank=10,
    niter=20,
)


In [4]:
# 💾 模型路径映射
modeltype2path = {
    "llama2-7b-chat-hf": "meta-llama/Llama-2-7b-chat-hf",
    "llama2-7b-hf": "meta-llama/Llama-2-7b-hf",
}

# ✅ 加载模型和 tokenizer
def get_llm(model_name, cache_dir):
    model = AutoModelForCausalLM.from_pretrained(
        modeltype2path[model_name],
        torch_dtype=torch.bfloat16,
        cache_dir=cache_dir,
        low_cpu_mem_usage=True,
        device_map="auto",
        token=os.environ.get("HF_TOKEN"),
    )
    model.seqlen = model.config.max_position_embeddings
    return model

model = get_llm(args.model, args.cache_dir)
model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_

In [5]:
# tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    modeltype2path[args.model],
    use_fast=False,
    cache_dir=args.cache_dir,
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model.resize_token_embeddings(len(tokenizer))

# device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Pruning (仅 unstructured 的 wanda 方法)
prune_n, prune_m = 0, 0
print("开始剪枝：", args.prune_method, args.prune_data)

Using device: cuda:0
开始剪枝： wanda GSM8K_direct


In [ ]:
prune_wanda(
    args=args,  # 不用 args
    model=model,
    tokenizer=tokenizer,
    model_base=args.model_base,  
    device=device,
    prune_n=prune_n,
    prune_m=prune_m,
    prune_data=args.prune_data,
)

# ✅ Sparsity 检查
sparsity_ratio_actual = check_sparsity(model)
print(f"实际 sparsity: {sparsity_ratio_actual:.6f}")

# # ✅ PPL 评估
# ppl = eval_ppl(
#     args=None,
#     model=model,
#     tokenizer=tokenizer,
#     device=device,
# )
# print(f"Perplexity (Wikitext): {ppl:.4f}")
# ✅ 选择性保存 mask（可选）
direction = "top" if args.neg_prune else "bottom"
print(f"保存 mask 方向: {direction}")
save_mask_flag = True
if save_mask_flag:
    mask = get_mask(model, args.neg_prune)
    mask_dir = os.path.join(save_dir, "FT_mask")
    os.makedirs(mask_dir, exist_ok=True)
    mask_path = os.path.join(mask_dir, f"mask_{direction}_{args.sparsity_ratio:.3f}.pt")
    torch.save(mask, mask_path)
    print(f"Mask 已保存: {mask_path}")


loading calibration data GSM8K_direct
dataset loading complete
prune every linear layer
pruning layer 0 name self_attn.q_proj
pruning layer 0 name self_attn.k_proj
pruning layer 0 name self_attn.v_proj
pruning layer 0 name self_attn.o_proj
pruning layer 0 name mlp.gate_proj
pruning layer 0 name mlp.up_proj
pruning layer 0 name mlp.down_proj
pruning layer 1 name self_attn.q_proj
pruning layer 1 name self_attn.k_proj
pruning layer 1 name self_attn.v_proj
pruning layer 1 name self_attn.o_proj
pruning layer 1 name mlp.gate_proj
pruning layer 1 name mlp.up_proj
pruning layer 1 name mlp.down_proj
pruning layer 2 name self_attn.q_proj
pruning layer 2 name self_attn.k_proj
pruning layer 2 name self_attn.v_proj
pruning layer 2 name self_attn.o_proj
pruning layer 2 name mlp.gate_proj
pruning layer 2 name mlp.up_proj
pruning layer 2 name mlp.down_proj
pruning layer 3 name self_attn.q_proj
pruning layer 3 name self_attn.k_proj
pruning layer 3 name self_attn.v_proj
pruning layer 3 name self_attn.o_

In [ ]:
save_mask_flag = True
if save_mask_flag:
    mask = get_mask(model, args.neg_prune)
    mask_dir = os.path.join(save_dir, "FT_mask")
    os.makedirs(mask_dir, exist_ok=True)
    mask_path = os.path.join(mask_dir, f"mask_top_{args.sparsity_ratio:.3f}.pt")
    torch.save(mask, mask_path)
    print(f"Mask 已保存: {mask_path}")

9.60% entries are True in mask.
